# Building AI API Services with Gemini

This notebook covers building production-ready AI APIs with Google Gemini integration. All code examples are meant to be copied to `.py` files and run as FastAPI servers.

## Learning Objectives
- Build AI chatbot API endpoints
- Integrate Google Gemini API
- Implement streaming responses
- Add error handling and retry logic
- Create complete, production-ready FastAPI applications

## 1. Gemini API Setup

### Installation

```bash
pip install fastapi uvicorn openai pydantic python-dotenv
```

### Environment Configuration

Create `.env` file:
```env
OPENAI_API_KEY=your-api-key-here
API_KEYS=your-custom-api-key-1,your-custom-api-key-2
```

### Basic Gemini Service

Create `services/gemini.py`:

```python
import openai
from typing import AsyncGenerator, Optional
import os
from dotenv import load_dotenv

load_dotenv()

class GeminiService:
    def __init__(self):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found in environment")
        
        API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=API_KEY)
        self.model = 'gpt-4o-mini'
    
    async def generate(self, prompt: str, **kwargs) -> str:
        """Generate text from prompt."""
        response = await self.model.generate_content_async(prompt, **kwargs)
        return response.choices[0].message.content
    
    async def generate_stream(self, prompt: str, **kwargs) -> AsyncGenerator[str, None]:
        """Stream text generation."""
        response = await self.model.generate_content_async(
            prompt,
            stream=True,
            **kwargs
        )
        
        async for chunk in response:
            if chunk.text:
                yield chunk.text

# Singleton instance
gemini_service = GeminiService()
```

## 2. Basic AI Chat Endpoint

Create `main.py` with basic chat functionality:

```python
from fastapi import FastAPI, HTTPException, Depends, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import List, Optional
import uuid
import time
from datetime import datetime
from services.gemini import gemini_service
import os
from dotenv import load_dotenv

load_dotenv()

app = FastAPI(
    title="AI Chat API",
    description="Backend API for AI-powered chat services",
    version="1.0.0"
)

# CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Models
class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=4096, description="User message")
    system_instruction: Optional[str] = Field(default=None, max_length=2048)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: Optional[int] = Field(default=None, ge=1, le=8192)

class ChatResponse(BaseModel):
    id: str
    response: str
    model: str
    tokens_used: Optional[int]
    latency_ms: float
    created_at: datetime

# API Key verification
async def verify_api_key(
    x_api_key: Optional[str] = Header(None, alias="X-API-Key")
):
    valid_keys = os.getenv("API_KEYS", "").split(",")
    if not x_api_key or x_api_key not in valid_keys:
        raise HTTPException(status_code=401, detail="Invalid API key")
    return x_api_key

# Endpoints
@app.get("/")
async def root():
    return {"message": "AI Chat API", "version": "1.0.0"}

@app.get("/health")
async def health():
    return {"status": "healthy", "model": "gemini-pro"}

@app.post("/chat", response_model=ChatResponse)
async def chat(
    request: ChatRequest,
    api_key: str = Depends(verify_api_key)
):
    start_time = time.time()
    
    try:
        # Build prompt with optional system instruction
        prompt = request.message
        if request.system_instruction:
            prompt = f"{request.system_instruction}\n\nUser: {request.message}"
        
        # Generate response
        response_text = await gemini_service.generate(
            prompt,
            generation_config={
                "temperature": request.temperature,
                "max_output_tokens": request.max_tokens
            }
        )
        
        latency = (time.time() - start_time) * 1000
        
        return ChatResponse(
            id=str(uuid.uuid4()),
            response=response_text,
            model="gemini-pro",
            tokens_used=len(response_text.split()),
            latency_ms=latency,
            created_at=datetime.now()
        )
        
    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"AI generation failed: {str(e)}"
        )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

## 3. Streaming Responses

### SSE (Server-Sent Events) Streaming

Add to `main.py`:

```python
from fastapi.responses import StreamingResponse
import json

class StreamChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=4096)
    system_instruction: Optional[str] = None
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: Optional[int] = Field(default=None, ge=1)

@app.post("/chat/stream")
async def chat_stream(
    request: StreamChatRequest,
    api_key: str = Depends(verify_api_key)
):
    async def event_generator():
        try:
            prompt = request.message
            if request.system_instruction:
                prompt = f"{request.system_instruction}\n\nUser: {request.message}"
            
            async for chunk in gemini_service.generate_stream(
                prompt,
                generation_config={
                    "temperature": request.temperature,
                    "max_output_tokens": request.max_tokens
                }
            ):
                # Send each chunk as SSE
                yield f"data: {json.dumps({'text': chunk})}\n\n"
            
            # Signal completion
            yield f"data: {json.dumps({'done': True})}\n\n"
            
        except Exception as e:
            yield f"data: {json.dumps({'error': str(e)})}\n\n"
    
    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no"
        }
    )
```

## 4. Error Handling and Retry Logic

### Retry Decorator

Create `utils/retry.py`:

```python
import asyncio
import functools
from typing import Type, Tuple, Optional
import logging

logger = logging.getLogger(__name__)

def retry_async(
    max_retries: int = 3,
    delay: float = 1.0,
    backoff_factor: float = 2.0,
    exceptions: Tuple[Type[Exception], ...] = (Exception,)
):
    """Async retry decorator with exponential backoff."""
    def decorator(func):
        @functools.wraps(func)
        async def wrapper(*args, **kwargs):
            last_exception = None
            
            for attempt in range(max_retries + 1):
                try:
                    return await func(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    
                    if attempt < max_retries:
                        wait_time = delay * (backoff_factor ** attempt)
                        logger.warning(
                            f"Attempt {attempt + 1} failed: {e}. "
                            f"Retrying in {wait_time}s..."
                        )
                        await asyncio.sleep(wait_time)
                    else:
                        logger.error(f"All {max_retries + 1} attempts failed")
            
            raise last_exception
        
        return wrapper
    return decorator
```

### Enhanced Gemini Service with Retry

Update `services/gemini.py`:

```python
import openai
from typing import AsyncGenerator, Optional
import os
from dotenv import load_dotenv
from utils.retry import retry_async
import logging

logger = logging.getLogger(__name__)
load_dotenv()

class GeminiService:
    def __init__(self):
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found in environment")
        
        API_KEY = os.getenv("OPENAI_API_KEY")
client = openai.OpenAI(api_key=API_KEY)
        self.model = 'gpt-4o-mini'
    
    @retry_async(
        max_retries=3,
        delay=1.0,
        backoff_factor=2.0,
        exceptions=(Exception,)
    )
    async def generate(self, prompt: str, **kwargs) -> str:
        """Generate text with retry logic."""
        try:
            response = await self.model.generate_content_async(prompt, **kwargs)
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Gemini generation failed: {e}")
            raise
    
    @retry_async(
        max_retries=2,
        delay=0.5,
        backoff_factor=1.5,
        exceptions=(Exception,)
    )
    async def generate_stream(self, prompt: str, **kwargs) -> AsyncGenerator[str, None]:
        """Stream text generation with retry logic."""
        try:
            response = await self.model.generate_content_async(
                prompt,
                stream=True,
                **kwargs
            )
            
            async for chunk in response:
                if chunk.text:
                    yield chunk.text
                    
        except Exception as e:
            logger.error(f"Gemini streaming failed: {e}")
            raise
    
    async def generate_with_timeout(
        self,
        prompt: str,
        timeout_seconds: float = 30.0,
        **kwargs
    ) -> str:
        """Generate with timeout."""
        import asyncio
        
        try:
            return await asyncio.wait_for(
                self.generate(prompt, **kwargs),
                timeout=timeout_seconds
            )
        except asyncio.TimeoutError:
            raise TimeoutError(f"Gemini generation timed out after {timeout_seconds}s")

gemini_service = GeminiService()
```

## 5. Complete Production-Ready App

### Project Structure

```
ai-chat-api/
├── main.py                 # Main application
├── services/
│   ├── __init__.py
│   └── gemini.py          # Gemini service
├── utils/
│   ├── __init__.py
│   └── retry.py           # Retry decorator
├── models/
│   ├── __init__.py
│   ├── requests.py        # Request models
│   └── responses.py       # Response models
├── dependencies.py        # Shared dependencies
├── .env                   # Environment variables
└── requirements.txt       # Dependencies
```

### Complete `main.py`

```python
from fastapi import FastAPI, HTTPException, Depends, Header, status
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse, JSONResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import uuid
import time
import json
from datetime import datetime
import os
from dotenv import load_dotenv

from services.gemini import gemini_service

load_dotenv()

app = FastAPI(
    title="AI Chat API",
    description="Production-ready AI chat service with Gemini",
    version="1.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==================== Models ====================

class ChatMessage(BaseModel):
    role: str = Field(..., pattern=r"^(user|assistant|system)$")
    content: str = Field(..., min_length=1, max_length=32000)

class ChatRequest(BaseModel):
    message: str = Field(..., min_length=1, max_length=4096, description="User message")
    history: List[ChatMessage] = Field(default_factory=list, description="Conversation history")
    system_instruction: Optional[str] = Field(default=None, max_length=2048)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: Optional[int] = Field(default=None, ge=1, le=8192)
    stream: bool = Field(default=False, description="Enable streaming response")

class ChatResponse(BaseModel):
    id: str
    response: str
    model: str
    tokens_used: Optional[int]
    latency_ms: float
    created_at: datetime

class ErrorResponse(BaseModel):
    error: str
    message: str
    status_code: int
    request_id: Optional[str]

# ==================== Dependencies ====================

VALID_API_KEYS = os.getenv("API_KEYS", "").split(",")

async def verify_api_key(
    x_api_key: Optional[str] = Header(None, alias="X-API-Key")
):
    if not x_api_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Missing API key. Include X-API-Key header."
        )
    
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=status.HTTP_403_FORBIDDEN,
            detail="Invalid API key"
        )
    
    return x_api_key

# ==================== Exception Handlers ====================

@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": exc.detail,
            "message": exc.detail,
            "status_code": exc.status_code
        }
    )

@app.exception_handler(Exception)
async def global_exception_handler(request, exc):
    return JSONResponse(
        status_code=500,
        content={
            "error": "Internal server error",
            "message": str(exc),
            "status_code": 500
        }
    )

# ==================== Endpoints ====================

@app.get("/")
async def root():
    return {
        "service": "AI Chat API",
        "version": "1.0.0",
        "docs": "/docs"
    }

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model": "gemini-pro",
        "timestamp": datetime.now().isoformat()
    }

@app.post("/chat", response_model=ChatResponse)
async def chat(
    request: ChatRequest,
    api_key: str = Depends(verify_api_key)
):
    """Synchronous chat endpoint."""
    request_id = str(uuid.uuid4())
    start_time = time.time()
    
    try:
        # Build prompt with history
        prompt_parts = []
        
        if request.system_instruction:
            prompt_parts.append(f"System: {request.system_instruction}")
        
        for msg in request.history:
            role = "User" if msg.role == "user" else "Assistant"
            prompt_parts.append(f"{role}: {msg.content}")
        
        prompt_parts.append(f"User: {request.message}")
        prompt = "\n".join(prompt_parts)
        
        # Generate response
        response_text = await gemini_service.generate_with_timeout(
            prompt,
            timeout_seconds=30.0,
            generation_config={
                "temperature": request.temperature,
                "max_output_tokens": request.max_tokens
            }
        )
        
        latency = (time.time() - start_time) * 1000
        
        return ChatResponse(
            id=request_id,
            response=response_text,
            model="gemini-pro",
            tokens_used=len(response_text.split()),
            latency_ms=latency,
            created_at=datetime.now()
        )
        
    except TimeoutError as e:
        raise HTTPException(
            status_code=status.HTTP_504_GATEWAY_TIMEOUT,
            detail=str(e)
        )
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"AI generation failed: {str(e)}"
        )

@app.post("/chat/stream")
async def chat_stream(
    request: ChatRequest,
    api_key: str = Depends(verify_api_key)
):
    """Streaming chat endpoint."""
    async def event_generator():
        request_id = str(uuid.uuid4())
        
        try:
            # Build prompt
            prompt_parts = []
            
            if request.system_instruction:
                prompt_parts.append(f"System: {request.system_instruction}")
            
            for msg in request.history:
                role = "User" if msg.role == "user" else "Assistant"
                prompt_parts.append(f"{role}: {msg.content}")
            
            prompt_parts.append(f"User: {request.message}")
            prompt = "\n".join(prompt_parts)
            
            # Send start event
            yield f"data: {json.dumps({'type': 'start', 'request_id': request_id})}\n\n"
            
            # Stream response
            async for chunk in gemini_service.generate_stream(
                prompt,
                generation_config={
                    "temperature": request.temperature,
                    "max_output_tokens": request.max_tokens
                }
            ):
                yield f"data: {json.dumps({'type': 'chunk', 'text': chunk})}\n\n"
            
            # Send completion event
            yield f"data: {json.dumps({'type': 'done', 'request_id': request_id})}\n\n"
            
        except Exception as e:
            yield f"data: {json.dumps({'type': 'error', 'message': str(e)})}\n\n"
    
    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream",
        headers={
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no"
        }
    )

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
```

## 6. Testing the Complete API

### Using curl

```bash
# Start the server
uvicorn main:app --reload --host 0.0.0.0 --port 8000

# Test basic chat
curl -X POST "http://localhost:8000/chat" \
  -H "Content-Type: application/json" \
  -H "X-API-Key: your-api-key" \
  -d '{"message": "Hello, AI!"}'

# Test streaming chat
curl -X POST "http://localhost:8000/chat/stream" \
  -H "Content-Type: application/json" \
  -H "X-API-Key: your-api-key" \
  -d '{"message": "Tell me a story"}' \
  --no-buffer

# Test with history
curl -X POST "http://localhost:8000/chat" \
  -H "Content-Type: application/json" \
  -H "X-API-Key: your-api-key" \
  -d '{
    "message": "What was my last question?",
    "history": [
      {"role": "user", "content": "What is AI?"},
      {"role": "assistant", "content": "AI is artificial intelligence..."}
    ]
  }'
```

### Using Python

```python
import requests
import json

API_URL = "http://localhost:8000"
API_KEY = "your-api-key"

headers = {
    "Content-Type": "application/json",
    "X-API-Key": API_KEY
}

# Basic chat
response = requests.post(
    f"{API_URL}/chat",
    headers=headers,
    json={"message": "Hello!"},
    timeout=30
)
print(response.json())

# Streaming chat
response = requests.post(
    f"{API_URL}/chat/stream",
    headers=headers,
    json={"message": "Tell me a story"},
    stream=True
)

for line in response.iter_lines():
    if line:
        data = line.decode('utf-8').replace('data: ', '')
        if data:
            print(json.loads(data))
```

## Key Takeaways

1. **Gemini integration** requires proper API key management and error handling
2. **Streaming responses** provide better UX for long-running AI generation
3. **Retry logic** handles transient failures gracefully
4. **Timeout handling** prevents hanging requests
5. **Structured errors** help API consumers debug issues
6. **Complete project structure** separates concerns for maintainability

## Next Steps
- Check `exercises/exercise-01.md` for hands-on practice
- Study deployment options in `resources.md`
- Explore Module 14 for production deployment patterns